In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LassoCV, LinearRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.base import TransformerMixin, BaseEstimator, clone
from IPython.display import Markdown
import seaborn as sns
import warnings
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LassoCV, LinearRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.base import TransformerMixin, BaseEstimator, clone
from formulaic import Formula
from sklearn.neural_network import MLPRegressor
from sklearn.neural_network import MLPClassifier
warnings.simplefilter('ignore')
np.random.seed(1234)

In [ ]:
#from google.colab import files
#uploaded = files.upload()

In [2]:
data = pd.read_stata('/home/onyxia/work/data/dataPrivatePublic.dta')

## Nettoyage de la base

In [3]:
from functions.data_cleaning import clean_data
df = clean_data(data)

Ici, je recrée toutes les dummys comme dans le code stata pour les variables X.

## Partie DML

Dans cette partie, nous essayons de retrouver les résulats du LATE en effectuant un DML.










Commençons par des régressions naïves.

In [4]:
#Nous commençons par travailler sur un sous-échantillon de df
sample_cveopp = df.loc[df["SAMPLE_CVEOPP"] == 1].copy()

In [5]:
#Les variables de contrôle
X_limited = ["North", "IdF", "salaireG", "duree_listes_horsAR", "Insertion", "Q2", "agegr56", "exper0", "agegr4655", "French", "agegr3645", "Interim", "ce2", "Q3", "EndInterim", "salaireB", "primo", "agegr2635", "nivetude4", "tempcomp", "nivetude3", "salaireD", "EconLayoff", "PersLayoff", "salaireC", "Q1", "African", "salaireE"]

#Le traitement, ici le traitement effectif
d1 = "acceptationCVE_6MOIS"
d2 = "acceptationOPP_6MOIS"

#L'outcome
y="EMPLOI_6MOIS"

Faisons une première régression naïve de y sur d1, d2.

In [6]:
# Baseline OLS
w = sample_cveopp["POIDS_PZ_6MOIS"].astype(float)
X = sm.add_constant(sample_cveopp[["acceptationCVE_6MOIS", "acceptationOPP_6MOIS"]])
y = sample_cveopp["EMPLOI_6MOIS"]

lm0 = sm.WLS(y, X, weights=w).fit(cov_type="HC1")
vc0 = lm0.cov_params()
print("b_CVE =", lm0.params["acceptationCVE_6MOIS"], "se =", lm0.bse["acceptationCVE_6MOIS"])
print("b_OPP =", lm0.params["acceptationOPP_6MOIS"], "se =", lm0.bse["acceptationOPP_6MOIS"])


b_CVE = 0.07059823190069886 se = 0.01835810764598337
b_OPP = -0.006245669278886501 se = 0.006917396148106212


Now, I estimate with a set of controls (which are that that play the most in term of imbalance between the treated and the non-treated).

In [7]:
# Regression on baseline controls
X_limited_corrected = [
    "North", "femme", "IdF", "salaireG", "Insertion", "Q2",
     "exper", "French", "age", "Interim", "ce2", "Q3", "EndInterim", "salaireB", "primo", "nivetude4",
    "tempcomp", "nivetude3", "salaireD", "EconLayoff", "PersLayoff",
    "salaireC", "Q1", "African", "salaireE", "duree_listes_horsAR", "marie", "nenf", "ce1",
]

# Ensure 'exper' is numeric before proceeding to prevent ValueError from statsmodels
sample_cveopp["exper"] = pd.to_numeric(sample_cveopp["exper"], errors="coerce")

varlist = [d1] + [d2] + X_limited_corrected
w = sample_cveopp["POIDS_PZ_6MOIS"].astype(float)

# Explicitly convert the DataFrame subset to float to prevent any object dtypes
X_df = sample_cveopp[varlist].astype(float)
X = sm.add_constant(X_df)
y = sample_cveopp["EMPLOI_6MOIS"]

lmC = sm.WLS(y, X, weights=w).fit(cov_type="HC1")
vcC = lmC.cov_params()
print("b_CVE =", lmC.params["acceptationCVE_6MOIS"], "se =", lmC.bse["acceptationCVE_6MOIS"])
print("b_OPP =", lmC.params["acceptationOPP_6MOIS"], "se =", lmC.bse["acceptationOPP_6MOIS"])

b_CVE = 0.05797098559873487 se = 0.014745623746022243
b_OPP = 0.028816756521897836 se = 0.00539588702048443


## First stage of the DML

Ici, l'idée est d'essayer de voir si, en partant de l'hypothèse de unconfoundedness, nous parvenons à retrouver un résultat moyen causal qui soit du même ordre que le résultat du LATE de l'article (résultat sur les compliers seulement). Pour se faire, nous estimons les paramètres $\alpha_1$ et $\alpha_2$ du modèle statistique partiellement linéaire suivant,


\begin{equation}
 Y = \alpha_1 \mathbb{1}(Z=1) +  \alpha_2 \mathbb{1}(Z=2) + + g(X) + \epsilon,
\end{equation}
 avec $$\quad E (\epsilon | \mathbb{1}(Z=1), \mathbb{1}(Z=2), X) = 0.$$

A traduire

For $\tilde Y = Y- E(Y|X)$ and $\tilde D= D- E(D|X)$, we can write
$$
\tilde Y = \alpha \tilde D + U, \quad E (U |\tilde D) =0.
$$
Parameter $\alpha$ is then estimated using cross-fitting approach to obtain the residuals $\tilde D$ and $\tilde Y$.
The algorithm comsumes $Y, D, X$, and machine learning methods for learning the residuals $\tilde Y$ and $\tilde D$, where
the residuals are obtained by cross-validation (cross-fitting).

The statistical parameter $\alpha$ has a causal interpretation of being the effect of $D$ on $Y$ in the causal DAG $$ D\to Y, \quad X\to (D,Y)$$ or the counterfactual outcome model with conditionally exogenous (conditionally random) assignment of treatment $D$ given $X$:
$$
Y(d) = d\alpha + g(X) + U(d),\quad  U(d) \text{ indep } D |X, \quad Y = Y(D), \quad U = U(D).
$$


### I define a transformer that constructs the engineered features for controls

In [8]:
X_limited_corrected = [
    "North", "femme", "IdF", "salaireG", "Insertion", "Q2",
     "exper", "French", "age", "Interim", "ce2", "Q3", "EndInterim", "salaireB", "primo", "nivetude4",
    "tempcomp", "nivetude3", "salaireD", "EconLayoff", "PersLayoff",
    "salaireC", "Q1", "African", "salaireE", "duree_listes_horsAR", "marie", "nenf", "ce1",
]
sample_cveopp["exper"] = pd.to_numeric(sample_cveopp["exper"], errors="coerce")
sample_cveopp[X_limited_corrected].describe()

,North,femme,IdF,salaireG,Insertion,Q2,exper,French,age,Interim,...,EconLayoff,PersLayoff,salaireC,Q1,African,salaireE,duree_listes_horsAR,marie,nenf,ce1
count,43977.000000,43977.000000,43977.000000,43977.000000,43977.000000,43977.000000,43977.0,43977.000000,43977.000000,43977.000000,...,43977.000000,43977.000000,43977.000000,43977.000000,43977.000000,43977.000000,43977.000000,43977.000000,43977.000000,43977.000000
mean,0.099757,0.500875,0.806990,0.101599,0.276804,0.353094,6.981672,0.813471,36.714942,0.359961,...,0.124724,0.408645,0.085658,0.171590,0.116970,0.174091,224.029373,0.462128,0.902244,0.229597
std,0.299679,0.500005,0.394665,0.302123,0.447424,0.477937,7.947843,0.389538,10.503770,0.479994,...,0.330410,0.491589,0.279862,0.377028,0.321388,0.379192,120.972954,0.498569,1.247945,0.420579
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,17.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-87.400002,0.000000,0.000000,0.000000
25%,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.0,1.000000,28.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,116.000000,0.000000,0.000000,0.000000
50%,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,4.0,1.000000,35.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,237.000000,0.000000,0.000000,0.000000
75%,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,10.0,1.000000,45.000000,1.000000,...,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,353.000000,1.000000,2.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,42.0,1.000000,65.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,365.000000,1.000000,9.000000,1.000000


In [9]:
class FormulaTransformer(TransformerMixin, BaseEstimator):

    def __init__(self, formula, array=False):
        self.formula = formula
        self.array = array

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        df = Formula(self.formula).get_model_matrix(X)
        if self.array:
            return df.values
        return df

In [10]:
transformer = FormulaTransformer(
    "0 "
    "+ poly(age, degree=3, raw=True)"
    "+ poly(exper, degree=3, raw=True)"
    "+ poly(duree_listes_horsAR, degree=3, raw=True)"
    "+ poly(nenf, degree=2, raw=True)"
    "+ North + IdF + French + African + femme + marie"
    "+ Interim + EndInterim + tempcomp"
    "+ ce1 + ce2"
    "+ nivetude3 + nivetude4"
    "+ salaireB + salaireC + salaireD + salaireE"
    "+ Q1 + Q2 + Q3"
    "+ EconLayoff + PersLayoff"
    "+ primo + Insertion"
    "+ age:exper"
    "+ femme:nenf"
    "+ femme:exper"
    "+ duree_listes_horsAR:exper"
    "+ French:IdF"
    "+ African:IdF"
    "+ nivetude3:exper"
    "+ nivetude4:exper",
    array=False
)

In [11]:
transformer.fit_transform(sample_cveopp[X_limited_corrected]).describe()

,"poly(age, degree=3, raw=True)[0]","poly(age, degree=3, raw=True)[1]","poly(age, degree=3, raw=True)[2]","poly(exper, degree=3, raw=True)[0]","poly(exper, degree=3, raw=True)[1]","poly(exper, degree=3, raw=True)[2]","poly(duree_listes_horsAR, degree=3, raw=True)[0]","poly(duree_listes_horsAR, degree=3, raw=True)[1]","poly(duree_listes_horsAR, degree=3, raw=True)[2]","poly(nenf, degree=2, raw=True)[0]",...,primo,Insertion,age:exper,femme:nenf,femme:exper,duree_listes_horsAR:exper,French:IdF,African:IdF,nivetude3:exper,nivetude4:exper
count,43977.000000,43977.000000,43977.000000,43977.000000,43977.000000,43977.000000,43977.000000,43977.000000,4.397700e+04,43977.000000,...,43977.000000,43977.000000,43977.000000,43977.000000,43977.00000,43977.000000,43977.000000,43977.000000,43977.000000,43977.000000
mean,36.714942,1458.313641,3164.318917,6.981672,111.910521,2497.422766,224.029373,64823.277344,2.057602e+07,0.902244,...,0.620529,0.276804,306.560998,0.484458,3.26357,1588.794993,0.635946,0.106715,2.222548,1.627578
std,10.503770,815.302898,19624.858003,7.947843,231.196534,7319.914763,120.972954,51300.066406,1.957052e+07,1.247945,...,0.485261,0.447424,416.496169,0.984388,6.24282,2229.955216,0.481169,0.308754,5.770111,5.206886
min,17.000000,289.000000,-32768.000000,0.000000,0.000000,0.000000,-87.400002,0.000000,-6.676277e+05,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,-1591.199974,0.000000,0.000000,0.000000,0.000000
25%,28.000000,784.000000,-14883.000000,1.000000,1.000000,1.000000,116.000000,13456.000000,1.560896e+06,0.000000,...,0.000000,0.000000,42.000000,0.000000,0.00000,167.999992,0.000000,0.000000,0.000000,0.000000
50%,35.000000,1225.000000,9261.000000,4.000000,16.000000,64.000000,237.000000,56169.000000,1.331205e+07,0.000000,...,1.000000,0.000000,135.000000,0.000000,0.00000,714.000000,1.000000,0.000000,0.000000,0.000000
75%,45.000000,2025.000000,19648.000000,10.000000,100.000000,1000.000000,353.000000,124609.000000,4.398698e+07,2.000000,...,1.000000,1.000000,396.000000,1.000000,4.00000,2072.000000,1.000000,0.000000,1.000000,0.000000
max,65.000000,4225.000000,31800.000000,42.000000,1764.000000,74088.000000,365.000000,133225.000000,4.862712e+07,9.000000,...,1.000000,1.000000,2480.000000,9.000000,42.00000,14965.000000,1.000000,1.000000,42.000000,42.000000


### Estimating the ATE

In [12]:
y = sample_cveopp["EMPLOI_6MOIS"].values
D1 = sample_cveopp['acceptationCVE_6MOIS'].values
D2 = sample_cveopp['acceptationOPP_6MOIS'].values
X = sample_cveopp[X_limited_corrected]

In [13]:
from functions.dml import dml_two_treatments
from functions.dml import summary_two_treatments

### Learners of E[Y|X], E[D1|X] and E[D2|X]

In [14]:
cv = KFold(n_splits=5, shuffle=True, random_state=123)

In [15]:
learners_d = {

    "Logit": make_pipeline(transformer, StandardScaler(), LogisticRegressionCV(cv=cv, penalty='l2', solver='liblinear')),

    "RF": make_pipeline(transformer, RandomForestClassifier(n_estimators=100, min_samples_leaf=10, ccp_alpha=.001)),

    "Tree": make_pipeline(transformer, DecisionTreeClassifier(min_samples_leaf=10, ccp_alpha=.001)),

    "GBF": make_pipeline(transformer, GradientBoostingClassifier(max_depth=2, n_iter_no_change=5)),

    "MLP": make_pipeline(StandardScaler(),
                       MLPRegressor(hidden_layer_sizes=(50, 50, 50, 50),
                                    activation='relu',
                                    solver='adam',
                                    alpha=0.0001,
                                    batch_size=200,
                                    learning_rate='constant',
                                    learning_rate_init=0.001,
                                    max_iter=200,
                                    shuffle=True,
                                    random_state=None,
                                    tol=1e-4,
                                    verbose=False,
                                    warm_start=False,
                                    momentum=0.9,
                                    nesterovs_momentum=True,
                                    early_stopping=True,
                                    validation_fraction=0.2,
                                    beta_1=0.9,
                                    beta_2=0.999,
                                    epsilon=1e-08,
                                    n_iter_no_change=10)
                       )
}

In [16]:
learners_y = {
    "Logit": make_pipeline(transformer, StandardScaler(), LogisticRegressionCV(cv=cv, penalty='l2', solver='liblinear')),

    "RF":  make_pipeline(transformer, RandomForestClassifier(n_estimators=100, min_samples_leaf=10, ccp_alpha=.001)),

    "Tree": make_pipeline(transformer, DecisionTreeClassifier(min_samples_leaf=10, ccp_alpha=.001)),

    "GBF": make_pipeline(transformer, GradientBoostingClassifier(max_depth=2, n_iter_no_change=5)),

    "MLP": make_pipeline(StandardScaler(),
                       MLPRegressor(hidden_layer_sizes=(50, 50, 50, 50),
                                    activation='relu',
                                    solver='adam',
                                    alpha=0.0001,
                                    batch_size=200,
                                    learning_rate='constant',
                                    learning_rate_init=0.001,
                                    max_iter=200,
                                    shuffle=True,
                                    random_state=None,
                                    tol=1e-4,
                                    verbose=False,
                                    warm_start=False,
                                    momentum=0.9,
                                    nesterovs_momentum=True,
                                    early_stopping=True,
                                    validation_fraction=0.2,
                                    beta_1=0.9,
                                    beta_2=0.999,
                                    epsilon=1e-08,
                                    n_iter_no_change=10)
                       )
}

### Estimate all combinations and compute goodness of fit

In [ ]:
results_summary = []
models = {}

for name_y, learner_y in learners_y.items():
    for name_d, learner_d in learners_d.items():

        print(f"Running combination: {name_y} / {name_d}")

        dml_out = dml_two_treatments(
            X=X,
            D1=D1,
            D2=D2,
            y=y,
            modely=learner_y,
            modeld1=learner_d,
            modeld2=learner_d,
            nfolds=5,
            classifier_y=True,
            classifier_d1=True,
            classifier_d2=True,
            cluster=False
        )

        (
            point1, point2, stderr1, stderr2,
            yhat, D1hat, D2hat,
            resy, resD1, resD2,
            epsilon
        ) = dml_out

        summ = summary_two_treatments(
            point1, point2, stderr1, stderr2,
            yhat, D1hat, D2hat, resy, resD1, resD2, epsilon,
            X, D1, D2, y,
            name1="CVE_treated", name2="OPP_treated"
        ).reset_index(names="treatment")

        # --- add separate RMSE columns ---
        rmse_d1 = np.sqrt(np.mean(resD1**2))
        rmse_d2 = np.sqrt(np.mean(resD2**2))

        summ["rmse D1"] = rmse_d1
        summ["rmse D2"] = rmse_d2

        summ["learner_y"] = name_y
        summ["learner_d"] = name_d


        results_summary.append(summ)

        models[f"{name_y}__{name_d}"] = {
            "summary": summ,
            "raw_output": dml_out
        }

results_summary = pd.concat(results_summary, ignore_index=True)

results_summary = results_summary.sort_values(
    by=["rmse y", "rmse D1", "rmse D2"],
    ascending=True
)


Running combination: Logit / Logit


DML steps: 100%|██████████| 5/5 [01:08<00:00, 13.73s/it]


Running combination: Logit / RF


DML steps: 100%|██████████| 5/5 [00:27<00:00,  5.60s/it]


Running combination: Logit / Tree


...

DML steps: 100%|██████████| 5/5 [00:23<00:00,  4.61s/it]


Running combination: RF / Logit


DML steps: 100%|██████████| 5/5 [00:51<00:00, 10.32s/it]


Running combination: RF / RF


DML steps:  20%|██        | 1/5 [00:05<00:20,  5.06s/it]

In [33]:
print("\nGoodness of fit and DML results for all combinations:\n")

print(
    results_summary[
        [
            "learner_y",
            "learner_d",
            "treatment",
            "rmse y",
            "rmse D1",
            "rmse D2",
            "estimate",
            "stderr"
        ]
    ].to_string(index=False)
)


Goodness of fit and DML results for all combinations:

learner_y learner_d   treatment   rmse y  rmse D1  rmse D2  estimate   stderr
      GBF      Tree CVE_treated 0.312958 0.163514 0.470245  0.089680 0.009183
      GBF      Tree OPP_treated 0.312958 0.163514 0.470245  0.040902 0.003193
      GBF        RF CVE_treated 0.313037 0.163513 0.470951  0.089557 0.009183
      GBF        RF OPP_treated 0.313037 0.163513 0.470951  0.040022 0.003188
      GBF     Logit CVE_treated 0.313124 0.306316 0.464799  0.088017 0.009213
      GBF     Logit OPP_treated 0.313124 0.306316 0.464799  0.037995 0.003236
      GBF       GBF CVE_treated 0.313126 0.163299 0.463501  0.088393 0.009208
      GBF       GBF OPP_treated 0.313126 0.163299 0.463501  0.039551 0.003244
      GBF       MLP CVE_treated 0.313199 0.165080 0.468360  0.085945 0.009108
      GBF       MLP OPP_treated 0.313199 0.165080 0.468360  0.037220 0.003210
    Logit       GBF CVE_treated 0.317424 0.163320 0.463537  0.090156 0.009332
    Logi

### Choose the best learner and compute the final estimation with the best combination

In [34]:
best_y = (
    results_summary.groupby("learner_y", as_index=False)["rmse y"]
    .mean()
    .sort_values("rmse y")
)

best_d1 = (
    results_summary.groupby("learner_d", as_index=False)["rmse D"]
    .mean()
    .sort_values("rmse D")
)

best_d2 = (
    results_summary.groupby("learner_d", as_index=False)["rmse D"]
    .mean()
    .sort_values("rmse D")
)

print("Best learner for Y:")
print(best_y.to_string(index=False))

print("\nBest learner for D1:")
print(best_d1.to_string(index=False))

print("\nBest learner for D2:")
print(best_d2.to_string(index=False))

best_learner_y  = best_y.iloc[0]["learner_y"]
best_learner_d1 = best_d1.iloc[0]["learner_d"]
best_learner_d2 = best_d2.iloc[0]["learner_d"]

print("\nSelected learners:")
print("Y  =", best_learner_y)
print("D1 =", best_learner_d1)
print("D2 =", best_learner_d2)

Best learner for Y:
learner_y   rmse y
      GBF 0.313089
    Logit 0.317424
     Tree 0.319182
       RF 0.319430
      MLP 0.320877

Best learner for D1:
learner_d   rmse D
      GBF 0.313390
      MLP 0.316514
     Tree 0.316878
       RF 0.317227
    Logit 0.385557

Best learner for D2:
learner_d   rmse D
      GBF 0.313390
      MLP 0.316514
     Tree 0.316878
       RF 0.317227
    Logit 0.385557

Selected learners:
Y  = GBF
D1 = GBF
D2 = GBF


In [35]:
dml_best = dml_two_treatments(
    X=X,
    D1=D1,
    D2=D2,
    y=y,
    modely=learners_y[best_learner_y],
    modeld1=learners_d[best_learner_d1],
    modeld2=learners_d[best_learner_d2],
    nfolds=5,
    classifier_y=True,
    classifier_d1=True,
    classifier_d2=True,
    cluster=False,
    progress=True
)

(
    point1, point2, stderr1, stderr2,
    yhat, D1hat, D2hat,
    resy, resD1, resD2,
    epsilon
) = dml_best

DML steps: 100%|██████████| 5/5 [00:17<00:00,  3.53s/it]


In [36]:
from scipy.stats import norm


def stars_from_pvalue(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    return ""


def make_dml_final_table(
    point1, point2, stderr1, stderr2,
    resy, resD1, resD2, epsilon,
    *,
    name1="CVE_treated",
    name2="OPP_treated",
    learner_y=None,
    learner_d1=None,
    learner_d2=None,
    n=None
):
    z1 = point1 / stderr1 if stderr1 > 0 else np.nan
    z2 = point2 / stderr2 if stderr2 > 0 else np.nan

    pval1 = 2 * (1 - norm.cdf(abs(z1))) if np.isfinite(z1) else np.nan
    pval2 = 2 * (1 - norm.cdf(abs(z2))) if np.isfinite(z2) else np.nan

    lower1 = point1 - 1.96 * stderr1
    upper1 = point1 + 1.96 * stderr1
    lower2 = point2 - 1.96 * stderr2
    upper2 = point2 + 1.96 * stderr2

    rmse_y = np.sqrt(np.mean(resy**2))
    rmse_d1 = np.sqrt(np.mean(resD1**2))
    rmse_d2 = np.sqrt(np.mean(resD2**2))
    rmse_final = np.sqrt(np.mean(epsilon**2))

    out = pd.DataFrame({
        "treatment": [name1, name2],
        "estimate": [point1, point2],
        "std.error": [stderr1, stderr2],
        "z": [z1, z2],
        "p.value": [pval1, pval2],
        "stars": [stars_from_pvalue(pval1), stars_from_pvalue(pval2)],
        "ci.lower": [lower1, lower2],
        "ci.upper": [upper1, upper2],
        "rmse_y": [rmse_y, rmse_y],
        "rmse_d1": [rmse_d1, rmse_d1],
        "rmse_d2": [rmse_d2, rmse_d2],
        "rmse_final": [rmse_final, rmse_final],
        "learner_y": [learner_y, learner_y],
        "learner_d1": [learner_d1, learner_d1],
        "learner_d2": [learner_d2, learner_d2],
        "n": [n, n]
    })

    return out

In [37]:
final_table = make_dml_final_table(
    point1, point2, stderr1, stderr2,
    resy, resD1, resD2, epsilon,
    name1="CVE_treated",
    name2="OPP_treated",
    learner_y=best_learner_y,
    learner_d1=best_learner_d1,
    learner_d2=best_learner_d2,
    n=len(y)
)

In [38]:
print("\nBest DML specification\n")
print(f"Y  learner  : {best_learner_y}")
print(f"D1 learner  : {best_learner_d1}")
print(f"D2 learner  : {best_learner_d2}")

print("\nFinal DML results\n")
print(final_table.to_string(index=False))


Best DML specification

Y  learner  : GBF
D1 learner  : GBF
D2 learner  : GBF

Final DML results

  treatment  estimate  std.error         z  p.value stars  ci.lower  ci.upper   rmse_y  rmse_d1  rmse_d2  rmse_final learner_y learner_d1 learner_d2     n
CVE_treated  0.088805   0.009209  9.643299      0.0   ***  0.070755  0.106854 0.313129 0.163275 0.463586    0.312366       GBF        GBF        GBF 43977
OPP_treated  0.039771   0.003243 12.262059      0.0   ***  0.033414  0.046128 0.313129 0.163275 0.463586    0.312366       GBF        GBF        GBF 43977


### Do the procedure for the variables EMPLOI_3MOIS, EMPLOI_9MOIS and EMPLOI_12MOIS to get the evolution of the causal effect

The results are really close to that to the LATE estimation ! This means that the DML procedure do work !

In [ ]:
#Code